# Task Statement 1.7: Manage Session State, Resumption, and Forking

> Domain 1 (27% of the exam) · Scenarios 2 (Code Generation) and 4 (Developer Productivity) · [Back to Domain 1](../README.md)

## In One Minute

- **What**: a session is a saved conversation. You can continue it, branch it, or replace it with a summary.
- **Resume**: `claude --resume <session-name>` (or `-r`) continues a named session; name one with `--name` / `-n`. In the Agent SDK, pass `resume=<session_id>`. Good when nothing important changed.
- **Fork**: `fork_session` (CLI: `--fork-session` with `--resume`; Python SDK: `fork_session=True`; TypeScript: `forkSession: true`) makes an independent branch from a shared baseline. A fork copies the conversation history, not the files on disk. Good for comparing approaches.
- **Stale results**: if files changed, tell the resumed session which ones. If the old tool results are mostly stale, start a **new session with a structured summary**.
- **Exam answer in one line**: resume when results are still valid, fork to compare, summarise into a fresh session when tool results are stale.

---

## 📌 Overview

### Why it matters
Work spans days. On Monday Claude read `db/conn.py` and saw a pool size of 5. On Tuesday you edited it. If you resume Monday's session and say nothing, Claude still believes the old file. It will suggest changes to code that no longer exists. The old tool results are **stale**.

### What the exam tests
- `--resume <session-name>` for continuing named work.
- `fork_session` for parallel, independent exploration from one baseline.
- **Informing** a resumed session which files changed.
- Starting a **new session with a structured summary** when tool results are stale.
- Choosing between resuming and starting fresh.

---

## 📖 Concepts Explained

### 1. The simple idea
Three tools, three jobs:

| Tool | Job |
|---|---|
| **Resume** | Keep going exactly where you were |
| **Fork** | Try two directions from the same starting point |
| **Fresh session + summary** | Start clean with only the facts you trust |

### 2. Resume (`--resume <session-name>`)
Use it when earlier tool results are still true. If a few files changed, **say so**. In [main.py](main.py), `SessionManager.resume(name, changed_files=[...])` appends a user message that lists the changed files and asks the session to re-read them before relying on earlier output. With no changes, it adds nothing.

### 3. Fork (`fork_session`)
Use it to compare approaches. Branch A and Branch B both start from the same analysis. Edits in A never leak into B. In [main.py](main.py), `SessionState.fork` deep-copies the messages and records the parent session id, so the parent and each fork stay independent.

### 4. Fresh session with a structured summary
Use it when many tool results are stale, for example after a big merge. Write down what is still **verified** (decisions, confirmed facts, what changed), then start a new session with that summary and the new instruction. `SessionManager.resume_with_injected_summary` does this: the new session contains only the summary and the current task, never the old stale tool output.

### 5. How to choose

| Situation | Choose |
|---|---|
| Short break. Nothing changed | Resume |
| A few files changed | Resume and name the changed files |
| Compare two designs from one analysis | Fork |
| Large changes or a long gap. Results stale | New session with a structured summary |

---

## 🧪 Worked Example

Monday: you analyse a legacy database layer in session `legacy_db_analysis`. Claude reads `db/conn.py` (pool size 5).

| Day | Event | Right move |
|---|---|---|
| Monday | Baseline analysis is done | Keep session `legacy_db_analysis` |
| Monday | You want to compare Async SQLAlchemy and Prisma | **Fork** twice. Each branch gets its own question. The parent is untouched |
| Tuesday | You edit `db/conn.py` and `db/pool.py` | `--resume legacy_db_analysis`, then tell it: "These files changed: `db/conn.py`, `db/pool.py`. Re-read them." |
| Next week | You merged forty upstream commits | **New session** with a summary: "Decision: Async SQLAlchemy 2.0. 50 sync models. Changed since: schema, pool config." Then the new task: "Produce the migration plan." |

Each choice matches how much of the old context is still true.

---

## 🔑 Key Terms

- **Session**: a saved conversation that can be continued.
- **`--resume <session-name>`**: continue a named session.
- **`fork_session`**: create an independent branch from a baseline.
- **Stale tool results**: earlier file or data reads that no longer match reality.
- **Structured summary**: the verified facts and changes, used to seed a fresh session.
- **Changed-files notice**: a message telling a resumed session which files changed.

---

## 🏗️ Architecture

```mermaid
flowchart TD
    Base["Baseline session<br>(analysis done)"] --> Choice{"What changed?"}

    Choice -->|"Compare two approaches"| Fork["fork_session"]
    Fork --> A["Branch A<br>(isolated)"]
    Fork --> B["Branch B<br>(isolated)"]

    Choice -->|"Nothing changed"| Resume["--resume session-name"]
    Choice -->|"A few files changed"| ResumeInform["--resume session-name<br>and list changed files"]

    Choice -->|"Many changes<br>(tool results stale)"| Fresh["New session with<br>structured summary"]
    Fresh --> Clean["Only verified facts<br>and the new task"]
```

---

## ⚠️ Exam Traps and Correct Patterns

| Topic | Correct pattern | Trap | Why the trap fails |
|---|---|---|---|
| **Compare designs** | Fork from the baseline | Try both in one thread | Ideas from A leak into B |
| **Small file changes** | Resume and name the changed files | Resume and say nothing | Claude trusts stale reads |
| **Large changes** | New session with a structured summary | Resume and hope | Stale tool results mislead the work |
| **Continuing named work** | `--resume <session-name>` | Paste a whole transcript into a new prompt | Wastes tokens and loses structure |

---

## ❌ Common Mistakes

- Resuming after big edits without a changed-files notice.
- Using one thread for two competing approaches.
- Writing a vague summary ("we looked at the database") instead of decisions and facts.
- Carrying old tool output into the fresh session, which defeats the purpose.
- Forking when the real need is just to continue.

---

## 💡 Quick Decision Rules

| Question | Answer |
|---|---|
| Nothing changed? | Resume |
| Some files changed? | Resume and list them |
| Many things changed, or results are stale? | Fresh session with summary |
| Two options to compare? | Fork |
| What goes in the summary? | Verified decisions, facts, what changed, the next task |

---

## 🧠 Claude 4.5 Notes

- Models: `claude-sonnet-4-5` for the session turns (`COORDINATOR_MODEL` in [main.py](main.py)), `claude-haiku-4-5` for simple workers.
- Extended thinking: `thinking={"type": "enabled", "budget_tokens": N}` with `N >= 1024` and `N < max_tokens`.
- With thinking on: no `temperature` (HTTP 400), no `top_k`, `top_p` only 0.95 to 1.0, and `tool_choice` only `auto` or `none`.
- When a session with thinking blocks is resumed, keep the assistant content unchanged, including `signature`. A fresh session starts with none.

---

## 📝 Exam-Style Practice Questions

**Q1.** After a long analysis session, a developer wants to compare two refactoring approaches that both start from that same analysis, without either approach influencing the other. What is the best way?

- A) Explore both in the same conversation, one after the other.
- B) Start two new sessions and redo the analysis in each.
- C) Resume the session and ask Claude to forget Approach A before starting B.
- D) Use `fork_session` to create two independent branches from the analysis baseline.

**Q2.** A developer resumes yesterday's session. Two files were edited overnight; everything else is the same. What is the best approach?

- A) Resume with `--resume <session-name>` and tell the session which two files changed so it re-reads them.
- B) Resume and say nothing. Claude will notice the differences.
- C) Delete the session and start without any summary.
- D) Fork the session and use the fork as is.

**Q3.** After merging forty upstream commits, a developer wants to continue an old investigation. Most earlier tool results no longer match the code. What is best?

- A) Resume and rely on the earlier tool results.
- B) Start a new session seeded with a structured summary of verified findings and the changes, plus the new task.
- C) Resume and ask Claude to guess which results are stale.
- D) Fork the old session and continue in the fork.

### Answers

**Q1: D.**
- A) The two approaches contaminate each other in one thread.
- B) Redoing the analysis twice wastes time and may give different baselines.
- C) Asking to "forget" is unreliable, and the earlier text is still in context.

**Q2: A.**
- B) Claude has no way to know that files changed outside the session.
- C) Throwing away the context loses useful work for a two-file change.
- D) A fork carries the same stale reads. It adds nothing for this case.

**Q3: B.**
- A) The old tool results are stale and would mislead the work.
- C) Claude cannot know which results are stale without re-reading everything.
- D) A fork still contains the stale tool results.

---

## ✅ Self-Check Checklist

- [ ] I can say when to use `--resume <session-name>`.
- [ ] I can say when to use `fork_session`.
- [ ] I know to tell a resumed session which files changed.
- [ ] I can explain when to start a new session with a structured summary.
- [ ] I know what makes a good summary: verified facts, decisions, changes, next task.

**Memory hook**: *"Resume if true, fork to compare, summarise if stale."*

---

## 🛠️ Hands-on Workbook Guide

Open [ts_1_7_session_management.ipynb](ts_1_7_session_management.ipynb) (offline, mock client):

1. **Baseline**: create the first session.
2. **Fork**: an independent branch that leaves the parent unchanged.
3. **Resume**: continue a named session and tell it which files changed.
4. **Fresh session with summary**: the recovery pattern for stale results.
5. **Checks**: forks stay isolated, the changed-files notice is added, and summaries carry only verified facts.

Code: [main.py](main.py) (live API, run manually only) · Offline tests: [tests/test_domain1_enhancements.py](../../tests/test_domain1_enhancements.py)

In [ ]:
# Step 1: Environment & Offline Setup
import copy
import sys
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 1.7 Session Management.")

### 🗃️ Step 2: Session State & Manager Implementation
We implement `SessionState` supporting branching/forking and `SessionManager` supporting summary injection.

In [ ]:
# Step 2: Session Manager Logic
@dataclass
class SessionState:
    session_id: str
    session_name: str
    messages: List[Dict[str, Any]] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def fork(self, new_name: str) -> "SessionState":
        """Branch from current session checkpoint into an isolated thread."""
        return SessionState(
            session_id=f"fork_{uuid.uuid4().hex[:8]}",
            session_name=new_name,
            messages=copy.deepcopy(self.messages),
            metadata={"parent_session_id": self.session_id},
        )


class SessionManager:
    def __init__(self) -> None:
        self.sessions: Dict[str, SessionState] = {}

    def create_session(self, name: str, initial_messages: Optional[List[Dict[str, Any]]] = None) -> SessionState:
        sid = f"sess_{uuid.uuid4().hex[:8]}"
        sess = SessionState(session_id=sid, session_name=name, messages=initial_messages or [])
        self.sessions[name] = sess
        return sess

    def resume(self, name: str, changed_files: Optional[List[str]] = None) -> SessionState:
        """--resume <session-name>: continue the same history and name any changed files."""
        sess = self.sessions[name]
        if changed_files:
            listed = "\n".join(f"- {p}" for p in changed_files)
            sess.messages.append({
                "role": "user",
                "content": "These files changed since your last tool results. "
                           f"Re-read them before relying on earlier output:\n{listed}",
            })
        return sess

    def resume_with_injected_summary(self, name: str, structured_summary: str, new_instruction: str) -> SessionState:
        """
        Stale Context Remedy: Discards bloated history and starts a clean session
        anchored with a structured summary of previous findings.
        """
        clean_messages = [
            {"role": "user", "content": f"### PREVIOUS ARCHITECTURAL CONTEXT\n{structured_summary}"},
            {"role": "assistant", "content": "Context acknowledged. Ready for next instructions."},
            {"role": "user", "content": new_instruction},
        ]
        return self.create_session(f"{name}_resumed", clean_messages)

print(" SessionManager declared with forking and summary injection.")

### 🧪 Step 3: Interactive Simulation
1. Create a base session.
2. Fork the session to test an alternative architecture in `branch_a`.
3. Verify that adding messages to `branch_a` leaves `base` unpolluted.
4. Mitigate stale context by resuming with an injected summary.
5. Resume a named session and tell it which files changed.

In [ ]:
# Step 3: Simulation
mgr = SessionManager()

# 1. Base session
base = mgr.create_session("base", [{"role": "user", "content": "Design database schema."}])
print(f"Created base session: {base.session_name} with {len(base.messages)} message(s)")

# 2. Fork into branch_a
branch_a = base.fork("branch_a")
branch_a.messages.append({"role": "user", "content": "Explore NoSQL option (MongoDB)"})
print(f"Forked {branch_a.session_name} with {len(branch_a.messages)} messages")
print(f"Base session message count remains: {len(base.messages)}")

# 3. Resume with injected summary
clean = mgr.resume_with_injected_summary(
    "base",
    structured_summary="Agreed schema: PostgreSQL with orders and items tables.",
    new_instruction="Add index on customer_id.",
)
print(f"\nClean resumed session created: {clean.session_name}")
print("Clean messages count:", len(clean.messages))

# 4. Resume a named session and tell it which files changed
mgr.create_session("work", [{"role": "user", "content": "Review db/conn.py"}])
resumed = mgr.resume("work", changed_files=["db/conn.py"])
print("Resume notice:", resumed.messages[-1]["content"])

### 🔬 Step 4: Verification Assertions
We assert that:
1. Forked session preserves parent history at fork time.
2. Mutating the child session does not affect the parent.
3. Resume with summary compresses context and clears stale messages.
4. Resuming with changed files adds a notice that names them.

In [ ]:
# Step 4: Verification Checks
assert branch_a.metadata["parent_session_id"] == base.session_id
assert len(branch_a.messages) == 2
assert len(base.messages) == 1, "Parent session must not be polluted by child fork!"

# Verify summary injection
assert "PREVIOUS ARCHITECTURAL CONTEXT" in clean.messages[0]["content"]
assert "Add index on customer_id" in clean.messages[2]["content"]

assert "db/conn.py" in resumed.messages[-1]["content"]

print(" All verification assertions passed! Session management verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Use `fork_session` for divergent exploration paths; never mutate the primary branch.
- **Rule 2**: When files or requirements change, discard long sessions and inject a **structured Markdown summary** to defeat the stale context trap.
- **Rule 3**: Do not use `--resume` indefinitely on large refactors; stale caches cause hallucinations.